# Read recorded results from MLflow

This notebook only reads completed compact MLflow evaluations. It never prepares data or trains models.
Set FINAL_RUN_ID to pin a final evaluation, or leave it empty to display the most recently completed final run.
Old hierarchical runs remain in MLflow and can be inspected in its UI; this reader uses the new compact format.


In [1]:
import os
import pandas as pd
from mlflow import MlflowClient
from election.models.tracking import read_evaluation
from IPython.display import display

FINAL_RUN_ID = ""  # Paste the final run ID printed by 00_run_pipeline.ipynb to pin a result.
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_NAME", "election-prediction")
client = MlflowClient(tracking_uri=tracking_uri)
if not FINAL_RUN_ID:
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError("No experiment found; configure MLflow or specify a final run ID.")
    runs = client.search_runs([experiment.experiment_id],
        filter_string="tags.record_type = 'evaluation-v1' and tags.purpose = 'final_evaluation' and tags.record_complete = 'true' and attributes.status = 'FINISHED'",
        order_by=["attributes.end_time DESC"], max_results=1)
    if not runs:
        raise ValueError("No completed compact final evaluation found.")
    FINAL_RUN_ID = runs[0].info.run_id
final = read_evaluation(FINAL_RUN_ID, tracking={"tracking_uri": tracking_uri, "experiment_name": experiment_name})
fold = final["outer_results"]["2024"]
display(pd.DataFrame([{
    "Selected model": final["model_id"],
    "2024 accuracy": fold["accuracy"],
    "Evaluated seats": fold["evaluation_rows"],
    "Final run": FINAL_RUN_ID,
    "Historical selection run": final["metadata"]["selection_run_id"],
}]))
print("Selected hyperparameters:", fold["hyperparameters"])
print("Refit durations:", fold["refit_durations"])


,Selected model,2024 accuracy,Evaluated seats,Final run,Historical selection run
0,conditional_xgboost,0.737342,632,8c30d51cdf4646efb4ecb004a02d38df,422d3a36ee624753bfbd38cad6e59cbe


Selected hyperparameters: {'challenger.max_depth': 2, 'challenger.n_estimators': 200, 'change.max_depth': 2, 'change.n_estimators': 50}
Refit durations: {}
